---
# **Suplementos Alimentares**
---


🎯 **Objetivo:** Verificar qual das três fórmulas de proteína em pó (Fórumula 1, 2 ou 3) proporciona maior ganho de massa muscular em atletas?


---
**Variáveis:**

- `id_produto`: Código identificador do suplemento (Fórmula 1, 2 ou 3). 
- `id_atleta`: Código identificador do atleta que participou do estudo. 
- `ganho_massa`: Quantidade de massa muscular ganha (em kg) após 8 semanas de uso. 
- `idade`: Idade do atleta. 
- `frequencia_treino`: Número médio de treinos semanais do atleta. 
---

Desafio Estatística com Python - Teste de hipóteses
Squad Nina da Hora | Bootcamp Data Analytics 2026.1

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import math
import numpy as np
import pandas as pd

from IPython.display import display, Markdown

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Estatística
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Configuração visual
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

paleta = "flare"
cores = sns.color_palette(paleta, n_colors=2)

In [ ]:
# ==============================
# IMPORTACAO DO DATASET 
# ==============================

arquivo = 'bd_suplementos'
url = f'https://raw.githubusercontent.com/Squad-Nina-da-Hora/wmc-desafio-suplementos/main/{arquivo}.csv'
df = pd.read_csv(url)

In [ ]:
# ==============================
# PERFIL DO DATASET
# ==============================

linhas = df.shape[0]
colunas = df.shape[1]
print(f"O dataset possui {linhas} linhas e {colunas} colunas.")

display(Markdown("---"))

info_df = pd.DataFrame({
    'Coluna': df.columns,
    'Tipo': df.dtypes.values,
    'Não nulos': df.count().values,
    'Nulos': df.isnull().sum().values
})

info_df

### Questão 1 — Análise Exploratória dos Dados (EDA)

In [ ]:
# Média e mediana do ganho de massa para cada suplemento

sup_massa = df.groupby('id_produto')['ganho_massa'].agg(['mean', 'median'])
sup_massa.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)']
sup_massa

In [ ]:
# Visual da distribuição

fig, axes = plt.subplots(3, 2, figsize=(14, 20))

produto = sorted(df['id_produto'].unique())

for i, supl in enumerate(produto):
  df_supl = df[df['id_produto'] == supl]
  cor_atual = sns.color_palette('flare', 3)[i]

  sns.histplot(data=df_supl, x='ganho_massa', kde=True, color=cor_atual, stat='count', ax=axes[i, 0])
  axes[i, 0].set_title(f'Histograma: {supl}')
  axes[i, 0].set_xlabel('Ganho de Massa (kg)')
  axes[i, 0].set_ylabel('Número de atletas')

  if axes[i, 0].containers:
      axes[i, 0].bar_label(axes[i, 0].containers[0], padding=3, fontsize=10, weight='bold')

  sns.boxplot(data=df_supl, y='ganho_massa',color=cor_atual, ax=axes[i, 1])
  sns.stripplot(data=df_supl, y='ganho_massa', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[i, 1])
  axes[i, 1].set_title(f'Boxplot: {supl}')
  axes[i, 1].set_ylabel('Ganho de Massa (kg)')
  axes[i, 1].set_xlabel(supl)

plt.tight_layout()
plt.show()

In [ ]:
# Outliers pelo método do IQR (geral)
q1 = df['ganho_massa'].quantile(0.25)
q3 = df['ganho_massa'].quantile(0.75)
iqr = q3 - q1

outliers = df[(df['ganho_massa'] < (q1 - 1.5 * iqr)) | (df['ganho_massa'] > (q3 + 1.5 * iqr))]
print(f"Total de outliers encontrados: {len(outliers)}")
outliers

# Fazendo análises extras com os dados do nossso dataframe
Seguindo essa ordem:

- Idade para cada Suplemento
- Frequência de treino para cada Suplemento
- Ganho de massa para frequência de treino
- Média e Mediana do ganho de massa por idade
- Média e Mediana do ganho de massa para o intervalo de idade criado
- Média e Mediana da frequência de treino por idade
- Média e Mediana da frequência de treino para o intervalo de idade criado


In [ ]:
# Média e mediana da idade para cada suplemento

sup_idade = df.groupby('id_produto')['idade'].agg(['mean', 'median'])
sup_idade.columns = ['Média de Idade', 'Mediana de Idade']
sup_idade

In [ ]:
# Visual da distribuição

fig, axes = plt.subplots(3, 2, figsize=(14, 20))

produto = sorted(df['id_produto'].unique())

for i, supl in enumerate(produto):
  df_supl = df[df['id_produto'] == supl]
  cor_atual = sns.color_palette('flare', 3)[i]

  sns.histplot(data=df_supl, x='idade', kde=True, color=cor_atual, stat='count', ax=axes[i, 0])
  axes[i, 0].set_title(f'Histograma: {supl}')
  axes[i, 0].set_xlabel('Idade (anos)')
  axes[i, 0].set_ylabel('Número de atletas')

  if axes[i, 0].containers:
      axes[i, 0].bar_label(axes[i, 0].containers[0], padding=3, fontsize=10, weight='bold')

  sns.boxplot(data=df_supl, y='idade',color=cor_atual, ax=axes[i, 1])
  sns.stripplot(data=df_supl, y='idade', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[i, 1])
  axes[i, 1].set_title(f'Boxplot: {supl}')
  axes[i, 1].set_ylabel('Idade')

plt.tight_layout()
plt.show()

In [ ]:
# Outliers pelo método do IQR (geral)
q1 = df['idade'].quantile(0.25)
q3 = df['idade'].quantile(0.75)
iqr = q3 - q1

outliers = df[(df['idade'] < (q1 - 1.5 * iqr)) | (df['idade'] > (q3 + 1.5 * iqr))]
print(f"Total de outliers encontrados: {len(outliers)}")
outliers

In [ ]:
# Média e mediana da frequência de treino para cada suplemento

sup_treino = df.groupby('id_produto')['frequencia_treino'].agg(['mean', 'median'])
sup_treino.columns = ['Média de Treinos', 'Mediana de Treinos']
sup_treino

In [ ]:
# Visual da distribuição

fig, axes = plt.subplots(3, 2, figsize=(14, 20))

produto = sorted(df['id_produto'].unique())

for i, supl in enumerate(produto):
  df_supl = df[df['id_produto'] == supl]
  cor_atual = sns.color_palette('flare', 3)[i]

  sns.histplot(data=df_supl, x='frequencia_treino', kde=True, color=cor_atual, stat='count', binwidth=1, ax=axes[i, 0])
  axes[i, 0].set_title(f'Histograma: {supl}')
  axes[i, 0].set_xlabel('Frequência')
  axes[i, 0].set_ylabel('Número de atletas')

  if axes[i, 0].containers:
      axes[i, 0].bar_label(axes[i, 0].containers[0], padding=3, fontsize=10, weight='bold')

  sns.boxplot(data=df_supl, y='frequencia_treino',color=cor_atual, ax=axes[i, 1])
  sns.stripplot(data=df_supl, y='frequencia_treino', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[i, 1])
  axes[i, 1].set_title(f'Boxplot: {supl}')
  axes[i, 1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

In [ ]:
# Outliers pelo método do IQR (geral)
q1 = df['frequencia_treino'].quantile(0.25)
q3 = df['frequencia_treino'].quantile(0.75)
iqr = q3 - q1

outliers = df[(df['frequencia_treino'] < (q1 - 1.5 * iqr)) | (df['frequencia_treino'] > (q3 + 1.5 * iqr))]
print(f"Total de outliers encontrados: {len(outliers)}")
outliers

In [ ]:
# Média e mediana do ganho de massa para a frequência de treino

freq_massa = df.groupby('frequencia_treino')['ganho_massa'].agg(['mean', 'median'])
freq_massa.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)']
freq_massa

In [ ]:
# Visual da distribuição por Frequência de Treino

# Descobre quantas frequências de treino diferentes existem para criar o número certo de linhas
frequencias_ordenadas = sorted(df['frequencia_treino'].unique())
num_linhas = len(frequencias_ordenadas)


fig, axes = plt.subplots(num_linhas, 2, figsize=(14, 6 * num_linhas))

for i, freq in enumerate(frequencias_ordenadas):
  df_freq = df[df['frequencia_treino'] == freq]
  cor_atual = sns.color_palette('flare', num_linhas)[i]


  sns.histplot(data=df_freq, x='ganho_massa', kde=True, color=cor_atual, stat='count', ax=axes[i, 0])
  axes[i, 0].set_title(f'Histograma - Treino: {freq}x por semana')
  axes[i, 0].set_xlabel('Ganho de Massa (kg)')
  axes[i, 0].set_ylabel('Número de atletas')

  if axes[i, 0].containers:
      axes[i, 0].bar_label(axes[i, 0].containers[0], padding=3, fontsize=10, weight='bold')

  sns.boxplot(data=df_freq, y='ganho_massa', color=cor_atual, ax=axes[i, 1])
  sns.stripplot(data=df_freq, y='ganho_massa', color='black', alpha=0.4, size=5, jitter=0.1, ax=axes[i, 1])
  axes[i, 1].set_title(f'Boxplot - Treino: {freq}x por semana')
  axes[i, 1].set_ylabel('Ganho de Massa (kg)')
  axes[i, 1].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Média e mediana do ganho de massa por idade

idade_massa = df.groupby('idade')['ganho_massa'].agg(['mean', 'median'])
idade_massa.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)']
idade_massa

In [ ]:
# Dividindo a quantidade de idade em intervalos

# 1. Definir as três condições lógicas
condicoes = [
    (df['idade'] <= 24),                              # De 18 a 24 anos
    (df['idade'] >= 25) & (df['idade'] <= 31),        # De 25 a 31 anos
    (df['idade'] >= 32)                               # De 32 a 39 anos
]

# 2. Definir os rótulos correspondentes para cada condição
rotulos = ['Jovens', 'Intermediário', 'Mais velhos']

# 3. Cria a coluna nova preenchida com um valor padrão
df['faixa_etaria'] = 'Não identificado'

for condicao, rotulo in zip(condicoes, rotulos):
    df.loc[condicao, 'faixa_etaria'] = rotulo

# 5. Verifica quantas pessoas caíram em cada grupo
df['faixa_etaria'].value_counts()

In [ ]:
# Média e mediana para o intervalo de idade criado

idade_ganho = df.groupby('faixa_etaria')['ganho_massa'].agg(['mean', 'median'])
idade_ganho.columns = ['Média de Ganho (kg)', 'Mediana de Ganho (kg)']
idade_ganho

In [ ]:
# Média e mediana da frequência de treino por idade

idade_treino = df.groupby('idade')['frequencia_treino'].agg(['mean', 'median'])
idade_treino.columns = ['Média de Treino', 'Mediana de Treino']
idade_treino

In [ ]:
# Média e mediana para o intervalo de idade criado

idade_freq = df.groupby('faixa_etaria')['frequencia_treino'].agg(['mean', 'median'])
idade_freq.columns = ['Média de Frequência', 'Mediana de Frequência']
idade_freq

### Questão 2 — Existe diferença estatisticamente significativa entre as fórmulas?

- $H_0$: Não há diferença entre as fórmulas.
- $H_1$: Há diferença significativa entre pelo menos duas fórmulas.

### Questão 3 — Correlação entre idade e ganho de massa muscular

### Questão 4 — Frequência de treino x ganho de massa (independente do suplemento)

### Questão 5 — Interação entre idade, frequência de treino e eficácia do suplemento

###  Questão 6 — Qual fórmula recomendar para atletas que treinam mais de 5x/semana?

### Visualizações - Gráficos

### Conclusão geral